In [101]:
import pandas as pd

In [102]:
pdf_form = pd.read_csv("data/input/form_response/form_response_18_06_2026.csv")

In [103]:
pdf_form = pdf_form.rename(columns={
    "Player 1 \r\nEnter full player name exactly as shown in the reference sheet (INSERT LINK)": "player_1",
    "Player 2": "player_2",
    "Player 3": "player_3",
    "Player 4": "player_4",
    "Player 5": "player_5",
    "Your Full Name": "name",
    "Timestamp": "timestamp"
})

In [104]:
pdf_form_long = pdf_form.melt(
    id_vars=["timestamp", "name"],
    value_vars=["player_1", "player_2", "player_3", "player_4", "player_5"],
    var_name="pick_slot",
    value_name="full_name"
)

In [ ]:
pdf_form_long = pdf_form_long.dropna(subset=["full_name"])
pdf_form_long["full_name"] = pdf_form_long["full_name"].str.strip()

In [107]:
pdf_form_long

,timestamp,name,pick_slot,full_name
0,18/06/2026 17:17:41,TEMP,player_1,Alessia Russo
1,18/06/2026 17:18:52,TEMP_2,player_1,Georgia Stanway
2,18/06/2026 17:17:41,TEMP,player_2,Lauren James
3,18/06/2026 17:18:52,TEMP_2,player_2,Vivianne Miedema
4,18/06/2026 17:17:41,TEMP,player_3,Beth Mead
5,18/06/2026 17:18:52,TEMP_2,player_3,Lauren Hemp
6,18/06/2026 17:17:41,TEMP,player_4,Lauren Hemp
7,18/06/2026 17:18:52,TEMP_2,player_4,Alessia Russo
8,18/06/2026 17:17:41,TEMP,player_5,Ella Toone
9,18/06/2026 17:18:52,TEMP_2,player_5,Keira Walsh


In [108]:
pdf_reference = pd.read_csv("data/input/player_reference/test.csv")

In [109]:
pdf_reference

,player_id,full_name,club,position,active
0,WSL_0001,Alessia Russo,Arsenal,FW,True
1,WSL_0002,Beth Mead,Arsenal,FW,True
2,WSL_0003,Stina Blackstenius,Arsenal,FW,True
3,WSL_0004,Caitlin Foord,Arsenal,FW,True
4,WSL_0005,Mariona Caldentey,Arsenal,MF,True
...,...,...,...,...,...
62,WSL_0063,Hawa Cissoko,West Ham United,DF,True
63,WSL_0064,Kristie Mewis,West Ham United,MF,True
64,WSL_0065,Alisha Lehmann,Aston Villa,FW,True
65,WSL_0066,Rachel Daly,Aston Villa,FW,True


In [110]:
pdf_goals = pd.read_csv("data/input/player_goals/GW_1.csv")

In [111]:
pdf_goals

,week,player_id,goals
0,GW_1,WSL_0001,2
1,GW_1,WSL_0002,1
2,GW_1,WSL_0003,1
3,GW_1,WSL_0007,2
4,GW_1,WSL_0008,1
5,GW_1,WSL_0009,1
6,GW_1,WSL_0010,1
7,GW_1,WSL_0011,2
8,GW_1,WSL_0012,2
9,GW_1,WSL_0015,2


In [112]:
df_prep = pdf_reference.merge(pdf_form_long, how="inner", on="full_name")



In [113]:
df_prep

,player_id,full_name,club,position,active,timestamp,name,pick_slot
0,WSL_0001,Alessia Russo,Arsenal,FW,True,18/06/2026 17:17:41,TEMP,player_1
1,WSL_0001,Alessia Russo,Arsenal,FW,True,18/06/2026 17:18:52,TEMP_2,player_4
2,WSL_0002,Beth Mead,Arsenal,FW,True,18/06/2026 17:17:41,TEMP,player_3
3,WSL_0012,Lauren James,Chelsea,FW,True,18/06/2026 17:17:41,TEMP,player_2
4,WSL_0018,Keira Walsh,Chelsea,MF,True,18/06/2026 17:18:52,TEMP_2,player_5
5,WSL_0024,Lauren Hemp,Manchester City,FW,True,18/06/2026 17:18:52,TEMP_2,player_3
6,WSL_0024,Lauren Hemp,Manchester City,FW,True,18/06/2026 17:17:41,TEMP,player_4
7,WSL_0026,Vivianne Miedema,Manchester City,FW,True,18/06/2026 17:18:52,TEMP_2,player_2
8,WSL_0035,Ella Toone,Manchester United,MF,True,18/06/2026 17:17:41,TEMP,player_5


In [114]:
a = df_prep.groupby("player_id")[["name"]].count().reset_index().rename(columns={"name": "num_picks"})

In [115]:
a

,player_id,num_picks
0,WSL_0001,2
1,WSL_0002,1
2,WSL_0012,1
3,WSL_0018,1
4,WSL_0024,2
5,WSL_0026,1
6,WSL_0035,1


In [116]:
df = df_prep.merge(pdf_goals, how="left", on="player_id").merge(a, how="left", on="player_id")
df["goals"] = df["goals"].fillna(0)

df["adjusted_goals "] = df["goals"] / df["num_picks"]

In [117]:
df

,player_id,full_name,club,position,active,timestamp,name,pick_slot,week,goals,num_picks,adjusted_goals
0,WSL_0001,Alessia Russo,Arsenal,FW,True,18/06/2026 17:17:41,TEMP,player_1,GW_1,2.0,2,1.0
1,WSL_0001,Alessia Russo,Arsenal,FW,True,18/06/2026 17:18:52,TEMP_2,player_4,GW_1,2.0,2,1.0
2,WSL_0002,Beth Mead,Arsenal,FW,True,18/06/2026 17:17:41,TEMP,player_3,GW_1,1.0,1,1.0
3,WSL_0012,Lauren James,Chelsea,FW,True,18/06/2026 17:17:41,TEMP,player_2,GW_1,2.0,1,2.0
4,WSL_0018,Keira Walsh,Chelsea,MF,True,18/06/2026 17:18:52,TEMP_2,player_5,GW_1,2.0,1,2.0
5,WSL_0024,Lauren Hemp,Manchester City,FW,True,18/06/2026 17:18:52,TEMP_2,player_3,GW_1,1.0,2,0.5
6,WSL_0024,Lauren Hemp,Manchester City,FW,True,18/06/2026 17:17:41,TEMP,player_4,GW_1,1.0,2,0.5
7,WSL_0026,Vivianne Miedema,Manchester City,FW,True,18/06/2026 17:18:52,TEMP_2,player_2,GW_1,2.0,1,2.0
8,WSL_0035,Ella Toone,Manchester United,MF,True,18/06/2026 17:17:41,TEMP,player_5,NaN,0.0,1,0.0


In [118]:
df.groupby("name")["goals"].sum().reset_index().sort_values(by="goals", ascending=False)

,name,goals
1,TEMP_2,7.0
0,TEMP,6.0
